In [7]:
import vr1
from vr1.core import FuelAssembly, Lattice
from vr1.settings import VR1Settings
from vr1.writer import WriterOpenMC
from vr1.plots import test_plots
from vr1.materials import VR1Materials
from vr1.VR1facility import Facility
import vr1.lattice_units as vlu
import openmc
from vr1.core import core_designs
import vr1.utils

# VR1 Tutorial

The first thing we will look at is understanding what each individual lattice unit looks like in OpenMC and how to import them. First, though, we will import our materials by running the cell below

In [8]:
openmc.Materials.cross_sections = "/Users/macris/openmc_data/endfb-viii.0-hdf5/cross_sections.xml" #must use viii.0 for C12
mats = VR1Materials()
mats_object = mats.get_materials() #generates materials obj
mats_object.export_to_xml()

settings = openmc.Settings()
settings.export_to_xml()

## Lattice Units

The VR1 package contains a module called "lattice_units" that holds all of the individual reactor elements that we saw in the slideshow. We will practice importing them here. These lattice unit objects can be made to return OpenMC Universes by calling the "*build*" method on them. If you don't recall what this means from earlier in the course, that's okay. A Universe is a geometry unit that contains a collection of Cell objects. An OpenMC Universe is sort of like a completed modular unit that contains a defined geometry. Think of it like this: a Cell object is an atom which can be used to construct molecules (Universes). You can even combine molecules to create larger molecules (Universes can be combined to make new Universes). Or don't think of it like that. Whatever works for you. 
<br> <br> <br> The class names for the units we have talked about are given in the table below: 

| Fuel Assembly | Dummy Element | Vertical Channel | Rabbit Tube | Control Rod | Grid Plate | 
| --- | --- | --- | --- | --- | --- |
| *IRT4M* | *Dummy* | *VertChannel* | *RabbitTube* | *AbsRod* | *GridPlate* |

<br>

Every fuel element takes an argument "*materials*" where you must pass the instance of VR1Materials that you've created to the lattice unit. In this tutorial notebook, we will practice forming the lattice unit objects (which are not net OpenMC Universes). In the next notebook, you will combine them to make lattices!


### Dummy

First, an example of now to create the simplest lattice unit will be provided below. The only argument this class takes is materials. To create this lattice unit, simply run the code below.

In [9]:
dummy = vlu.Dummy(materials=mats)

### Fuel Assembly
Next we will look at creating some fuel assemblies. There are three kinds: 4-, 6-, and 8-plate assemblies. The *IRT4M* class contains a keyword argument to deal with this: "*fa_type*". This can be set to a string containing '4', '6', or '8', accordingly. Try it below.

In [12]:
assembly_4plate = vlu.IRT4M(materials=mats,fa_type='4')
assembly_6plate = vlu.IRT4M(materials=mats,fa_type='6')
assembly_8plate = vlu.IRT4M(materials=mats,fa_type='8')

### Vertical Channel

Like with the fuel assemblies, there are a few variations of the VertChannel class that can be chosen by passing the correct argument. The keyword to specify the size of the vertical channel is simply "*diameter*". This takes an integer in units of milimeters. See if you can create the four types of vertical channel: 12mm, 25mm, 30mm, and 56mm. 

In [14]:
vertchannel_12mm = vlu.VertChannel(materials=mats,diameter=12)
vertchannel_25mm = vlu.VertChannel(materials=mats,diameter=25)
vertchannel_30mm = vlu.VertChannel(materials=mats,diameter=30)
vertchannel_56mm = vlu.VertChannel(materials=mats,diameter=56)

### GridPlate

The GridPlate object is another that takes only the materials as an input. The GridPlate class only creates an object that contains the grid plate for one lattice unit. That is, in the VR1 there is an 8x8 grid of possible locations for lattice units. The GridPlate class just represents a single entry of that grid. 

<br><br>You should have the hang of it now, so create a GridPlate unit in the code below with no help.

In [13]:
gridplateguy = vlu.GridPlate(materials=mats)

# Nested Units

There are a couple of lattice units that can be inserted into other units. These are: *RabbitTube*, *AbsRod*, and *VertChannel*. In the real VR1, the fuel elements are deliberately designed in such a way that they can support insertion of instruments or lattice units. In fact, control rods need to be inserted into a fuel element. $^*$ 
We will now learn how this works in the VR1-openmc repository.  


$^*$ so far as I could tell

### Absorption Rods

There are three types of absorption rod in the VR1: 

* Safety rods
* Experimental rods
* Control rods

They are identical in composition and size and their names differ only to indicate their functions. To create an *AbsRod* lattice unit, you have to specify the type of assembly it can be inserted into. This is done by passing the argument "*assembly_type*". The valid assembly types are: 

* '4'
* '6'
* 'd'

You are also able to change the height of the rod with the argument "*rod_height*". This indicates the removal height of the rod in cm. Passing a height of 10 thus indicates that the rod has been pulled out 10cm.

<br>

An example of an *AbsRod* object in a 6-plate assembly is given below. 

In [ ]:
absorption_rod_6plate = vlu.AbsRod(materials=mats, assembly_type='6')

Next, try making:

* absorption rod in a dummy fuel element
* absorption rod in a 4-plate assembly, removed 40 cm

In [15]:
absorption_rod_dummy = vlu.AbsRod(materials=mats, assembly_type='d')
absorption_rod_4plate = vlu.AbsRod(materials=mats, assembly_type='4', rod_height= 40.0)

### Inserting VertChannels

You can do the same thing with vertical channels that you can with absorption rods. Unlike absorption rods, though, vertical channels do not *need* to be inside of a fuel element, they simply can be. The argument passed to the *VertChannel* class to indicate which unit it should be inserted into is the same as for *AbsRod*: *lattice_type*. 

<br>

Try it below. Insert a 25mm diameter vertical channel into a 6-plate fuel assembly.

In [17]:
vertchannel_25_6plate = vlu.VertChannel(materials=mats, lattice_type='6')

# Plotting with openmc-plotter

Staring at these objects isn't nearly as fun as seeing their visualization. The package we use to plot our VR1 configurations is openmc-plotter. It is a very convenient visualization tool that we have written a function to easily call. It does require three xml files to exist in the directory it is being called from, however: "*materials.xml*," "*geometry.xml*," and, for some reason, "*settings.xml*". You should understand what these are from earlier in the class.

<br>

Below is an example code that builds an 8-plate assembly, initializes it as the root geometry, and plots it.

In [18]:
fuel_assembly_8plate = vlu.IRT4M(materials=mats, fa_type = '8')

universe_8plate = fuel_assembly_8plate.build()

geometry = openmc.Geometry(root=universe_8plate)
geometry.export_to_xml()

import vr1.utils
vr1.utils.plot_vr1()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

KeyboardInterrupt: 